# Problem Ranking System

[LeetCodeBot](https://github.com/xDDoubleTea/LeetCodeBot)

This notebook implements the scoring and ranking from the proposal. The
proposal's text is kept as written, section by section; cells marked
**Implementation note** say where the code had to interpret or depart from it.

Run it with `uv run jupyter lab` from this directory, then *Run → Run All Cells*.

## `/suggest` behavior

```
/suggest [mode] [filters...]
```

`/suggest` replies with a ranked list of problems, presented like `/filter-by-tag`:

1. The bot defers, scores every problem left after the filters for the invoking
   user, and sorts them by $\theta$ (see [Ranking](#Ranking)).
2. It replies with an embed listing the ranking, 10 problems per page, with
   buttons for the first, previous, next and last page.
3. Below the embed, a select menu lists the current page's problems as
   `id. title [Difficulty]`. Choosing one opens the problem's thread, creating it
   if it does not exist, exactly as `/problem <id>` does, and replies with a link
   to it.
4. The buttons and the menu stop working after 180 seconds.

- `mode` is `focus` (the default) or `mix`; see [Modes](#Modes:-focus-and-mix).
  Focus suggests problems similar to the ones the user solved most recently, mix
  suggests problems unrelated to them.
- `filters` are the command filters below.
- The user must have linked their LeetCode account with `/link`. Otherwise the
  command replies with the usual "You haven't linked your discord to leetcode
  account yet!" message.
- The reply is public, like `/filter-by-tag`, so the channel sees what was
  suggested.

## Goal

Rank the problems that the user might want to attempt next when invoking this command.

Therefore the score $\theta_i$ should represent "how likely the user will want to attempt problem $i$".

## Command

```
/suggest [filters...]
```

Notes:

- The user must have their leetcode account linked.

### Command Filters

- Database questions
- Premium
- AC'ed

<div class="alert alert-block alert-info">

**Implementation note.** The *Database questions* and *Premium* filters are the `INCLUDE_DATABASE` and
`INCLUDE_PREMIUM` settings below. *AC'ed* is not implemented: LeetCode returns
`status: null` for every problem unless the request carries that user's session
cookie, so another account's solved set is not available from the public API. The
exception is the last 20 accepted problems, which are public; the modes below
leave those out of the list.

</div>

## Staying close to what the user knows

The default ranking recommends problems near what the user already solves: the
same tags, the same difficulty, and in focus mode the same topic as the last few
problems. That is deliberate.

- **Context switching ends sessions.** Jumping between unrelated topics makes it
  easy to lose track of what you were doing, perform worse, and then quit to do
  something else. Performing worse is fine, since that is what learning looks
  like; quitting is the problem this ranking tries to remove.
- **Mastery.** Working through closely related problems builds understanding of
  one concept, or a few similar ones, before moving on.
- **It moves on by itself.** Problems are finite. Solving the closely related
  ones exhausts them, and the solved counts the ranking reads shift, so the
  recommendations follow.

Mix mode is the counterpart, for when variety is what the user wants. This is
from experience so far, with no user data behind it yet.

## Settings

In [1]:
import json
import sqlite3
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

from desc_parser import PREVIEW_LEN, parse_problem_desc

DB_PATH = Path("~/programming/LeetCodeBot/db/testbot.db").expanduser()
CACHE_DIR = Path("cache")

USERNAME = "xDDoubleTea"
REFERENCE_USERS = ["seanyu0425", "wytsai7660"]

W1, W2, W3 = 20, 10, 1
W_R = 20
W4 = 10
LIFT_SMOOTHING = 1
RECENT_K = 5
L = PREVIEW_LEN
M = -1 / 4500

INCLUDE_PREMIUM = False
INCLUDE_DATABASE = True
REFRESH_AC_RATE = False
SEED = 0
TIE_EPS = 0.05
TOP_N = 20

SESSION_GAPS_MIN = [15, 30, 60]
PERMUTATIONS = 10_000

pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

## Data

In [2]:
GRAPHQL_URL = "https://leetcode.com/graphql"
HEADERS = {
    "Content-Type": "application/json",
    "Referer": "https://leetcode.com",
    "User-Agent": "Mozilla/5.0",
}


def gql(query: str, variables: dict | None = None) -> dict:
    """POST one GraphQL query to LeetCode and return its `data`."""
    body = json.dumps({"query": query, "variables": variables or {}}).encode()
    request = urllib.request.Request(GRAPHQL_URL, data=body, headers=HEADERS)
    with urllib.request.urlopen(request, timeout=30) as response:
        payload = json.load(response)
    if payload.get("errors"):
        raise RuntimeError(payload["errors"][0]["message"])
    return payload["data"]

### Problems, from the bot's database

In [3]:
DIFFICULTY = {0: "Easy", 1: "Medium", 2: "Hard"}

with sqlite3.connect(DB_PATH) as db:
    problems = pd.read_sql_query(
        "select id, problem_frontend_id as frontend_id, title, url, difficulty, "
        "premium, description from problems",
        db,
    )
    problem_tags = pd.read_sql_query(
        "select a.problem_id as id, t.tag_name as tag from problem_tags a "
        "join topic_tags t on t.id = a.tag_id",
        db,
    )

tags_by_problem = problem_tags.groupby("id")["tag"].agg(lambda s: sorted(s))
problems["tags"] = problems["id"].map(tags_by_problem)
problems["tags"] = problems["tags"].apply(lambda t: t if isinstance(t, list) else [])
problems["difficulty"] = problems["difficulty"].map(DIFFICULTY)
problems["premium"] = problems["premium"].astype(bool)
problems["slug"] = problems["url"].str.rsplit("/", n=1).str[-1]
problems = problems.set_index("frontend_id").sort_index()

print(f"{len(problems)} problems, {problems['premium'].sum()} premium")
problems[["title", "difficulty", "premium", "tags"]].head()

4086 problems, 786 premium


,title,difficulty,premium,tags
frontend_id,,,,
1,Two Sum,Easy,False,"[Array, Hash Table]"
2,Add Two Numbers,Medium,False,"[Linked List, Math, Recursion]"
3,Longest Substring Without Repeating Characters,Medium,False,"[Hash Table, Sliding Window, String]"
4,Median of Two Sorted Arrays,Hard,False,"[Array, Binary Search, Divide and Conquer]"
5,Longest Palindromic Substring,Medium,False,"[Dynamic Programming, Manacher, String, Two Pointers]"


### AC rate, from LeetCode

<div class="alert alert-block alert-info">

**Implementation note.** The bot's queries do not request `acRate`, so the database has no AC rate. This
cell fetches it with the same paginated `questionList` query the bot's refresh
uses (about 41 requests, 0.3 s apart) and caches the result in
`cache/ac_rate.json`. Set `REFRESH_AC_RATE = True` to fetch again.

</div>

In [4]:
AC_RATE_QUERY = """
query ($skip: Int!) {
  questionList(categorySlug: "", limit: 100, skip: $skip, filters: {}) {
    totalNum
    data { questionFrontendId acRate }
  }
}
"""


def fetch_ac_rates() -> dict[int, float]:
    rates: dict[int, float] = {}
    skip, total = 0, None
    while total is None or skip < total:
        page = gql(AC_RATE_QUERY, {"skip": skip})["questionList"]
        total = page["totalNum"]
        for q in page["data"]:
            rates[int(q["questionFrontendId"])] = q["acRate"]
        skip += 100
        time.sleep(0.3)
    return rates


ac_rate_cache = CACHE_DIR / "ac_rate.json"
if REFRESH_AC_RATE or not ac_rate_cache.exists():
    CACHE_DIR.mkdir(exist_ok=True)
    ac_rate_cache.write_text(json.dumps(fetch_ac_rates()))

ac_rates = {int(k): v for k, v in json.loads(ac_rate_cache.read_text()).items()}
problems["ac_rate"] = problems.index.map(ac_rates)
print(f"AC rate known for {problems['ac_rate'].notna().sum()} of {len(problems)} problems")

AC rate known for 4086 of 4086 problems


### Description length

<div class="alert alert-block alert-info">

**Implementation note.** $\ell_i$ is the length of `parse_problem_desc`'s output, the text an embed
carries, using the copy in `desc_parser.py`. It is capped near
`PREVIEW_LEN` = 4000, which is what makes $L = 4000$ the maximum.

</div>

In [5]:
problems["length"] = problems["description"].fillna("").map(
    lambda html: len(parse_problem_desc(html))
)
problems["length"].describe()

count    4086.000
mean     1066.525
std       758.915
min        25.000
25%       597.000
50%      1038.000
75%      1497.750
max      4007.000
Name: length, dtype: float64

### Signed-in history (optional)

`USERNAME`'s full submission history, written to `cache/submissions.json` by
`uv run leetcode_auth.py` with a signed-in session. Variants use it to count
already-solved suggestions, and [Do switches end sessions?](#Do-switches-end-sessions?)
analyses it. Without the file, those parts skip.

In [6]:
HISTORY_PATH = CACHE_DIR / "submissions.json"
HAS_HISTORY = HISTORY_PATH.exists()

if HAS_HISTORY:
    submissions = pd.DataFrame(json.loads(HISTORY_PATH.read_text()))
    submissions = submissions.sort_values("timestamp", ignore_index=True)
    DIFFICULTY_BY_SLUG = dict(zip(problems["slug"], problems["difficulty"]))
    print(
        f"{len(submissions)} submissions, {submissions['slug'].nunique()} problems, "
        f"{(submissions['status'] == 'Accepted').sum()} accepted"
    )
else:
    print(f"{HISTORY_PATH} not found; run `uv run leetcode_auth.py` to fetch it")

777 submissions, 242 problems, 552 accepted


## Score of a Problem

<div class="alert alert-block alert-success">

For problem $i$, the score is defined by

$$
\theta_i = \sum_{j=1}^{C}c_j(i)w_j
$$

where

- $c_j(i)$ is an "evaluation" of the criterion $j$, possibly associated with problem $i$,
- $w_j$ is the weight of the criterion $j$,
- $C$ is the total number of criteria.

Note on the range of $c_j$ and $w_j$:

$$
\begin{cases}0\le c_j\le 1\\w_j\in\mathbb{N}\cup\{0\}\end{cases}
$$
for each $1\le j\le C$.

> not sure what evaluation should mean yet.
> maybe an estimate? [Estimation statistics](https://en.wikipedia.org/wiki/Estimation_statistics)
> [Estimator](https://en.wikipedia.org/wiki/Estimator)

</div>

## Criteria

### 1 Skill stats

- Tags that show up more frequently gets higher score.

<div class="alert alert-block alert-success">

Tag score for problem $i$:

$$
s_i=\dfrac{\sum_{k=1}^{|T|}f_k\cdot\mathrm{I}(t_k)}{\sum_{k=1}^{|T|}f_k}
$$

where

- $T$ is the set of all of the tags that show up in skills
- $f_k$ is the frequency of tag $k$ in $T$
- $t_k$ is tag $k$
- $\mathrm{I}(t_k)=1$ if problem $i$ has $t_k$ as a tag, $0$ otherwise.

The range of $s_i$ is $[0,1]$, since the denominator is clearly greater or equal to numerator.

</div>

- Maybe $c_1(i)=s_i$?

<details>
<summary>Skills example</summary>

![image](https://hackmd.io/_uploads/Bk8T_fZ2Gl.png)

</details>

<div class="alert alert-block alert-info">

**Implementation note.** $c_1(i) = s_i$. $f_k$ is `problemsSolved` from the profile's skills section
(`tagProblemCounts`), all three levels together. Only 47 of LeetCode's tags ever
appear there, so a tag outside that set contributes nothing to $s_i$.

</div>

In [7]:
SKILLS_QUERY = """
query ($username: String!) {
  matchedUser(username: $username) {
    tagProblemCounts {
      advanced { tagName problemsSolved }
      intermediate { tagName problemsSolved }
      fundamental { tagName problemsSolved }
    }
  }
}
"""


def fetch_skills(username: str) -> pd.DataFrame:
    """One row per tag in the user's skills section, with its level and f_k."""
    counts = gql(SKILLS_QUERY, {"username": username})["matchedUser"]["tagProblemCounts"]
    rows = [
        {"tag": t["tagName"], "level": level, "f": t["problemsSolved"]}
        for level, tags in counts.items()
        for t in tags
    ]
    return pd.DataFrame(rows).sort_values("f", ascending=False, ignore_index=True)


skills = {user: fetch_skills(user) for user in [USERNAME, *REFERENCE_USERS]}
skills[USERNAME].head(10)

,tag,level,f
0,Array,fundamental,101
1,String,fundamental,62
2,Math,intermediate,52
3,Hash Table,intermediate,48
4,Tree,intermediate,28
5,Bit Manipulation,intermediate,27
6,Two Pointers,fundamental,27
7,Sorting,fundamental,26
8,Binary Tree,intermediate,26
9,Depth-First Search,intermediate,24


### Recent AC

- Check Recent AC problems tags
- Maybe too hard, come back to this later.

<div class="alert alert-block alert-info">

**Implementation note.** Used by the two modes in
[Modes: focus and mix](#Modes:-focus-and-mix), not by the base ranking.
LeetCode's `recentAcSubmissionList` returns a user's last 20 accepted problems,
newest first, without login. The recent similarity of problem $i$ compares its
tags with the tags of each of the user's last $K$ accepted problems that have
tags, $p_1, \dots, p_K$:

$$
r_i = \frac{1}{K}\sum_{j=1}^{K} \frac{|T_i \cap T_{p_j}|}{|T_i \cup T_{p_j}|}
$$

where $T_i$ is the set of tags of problem $i$ (all tags, not only skill tags),
and a pair with no tags at all counts as 0. $r_i$ is in $[0, 1]$: 1 when every
recent problem has exactly problem $i$'s tags.

Untagged problems are skipped when picking $p_1, \dots, p_K$. LeetCode leaves
contest problems untagged for a while after the contest, so a user who just did
a contest would otherwise get $r_i = 0$ for every problem, and both modes would
fall back to the base ranking.

</div>

### 2 Problem AC rate

- higher -> evaluation higher
- Raw number (e.g. 73.5% -> 0.735) as the criterion "evaluation"

$$
c_2(i)=\text{Percentage raw number}/100
$$

### 3 Description length

- longer -> evalutaion lower

<div class="alert alert-block alert-success">

Simple linear function

$$
c_3(i)=m\ell_i+1
$$

where

- $\ell_i$ is the length of the description of problem $i$
- $m>-1/L$, where $L$ is the maximum possible description length (i.e., `4000` because of the cap `PREVIEW_LEN`).

</div>

> Maybe try $m=-1/4500$?

<div class="alert alert-block alert-info">

**Implementation note.** With $m<0$, $c_3(i)=m\ell_i$ is negative, which breaks $0\le c_j\le 1$. The
code uses

$$
c_3(i) = 1 + m\,\min(\ell_i, L), \qquad m = -1/4500
$$

which falls as the description grows and stays in $[1/9,\,1]$. Adding 1 to
every problem's $c_3$ adds the same $w_3$ to every $\theta_i$, so the ranking
is identical to $c_3(i)=m\ell_i$. `min` absorbs the `...` that truncation
appends, which can push $\ell_i$ slightly past 4000.

</div>

### Difficulty

- As a tie breaker, prefer medium -> easy -> hard.

### Last tie breaker

- randomly shuffle?

<div class="alert alert-block alert-info">

**Implementation note.** $\theta$ is a float, so two problems almost never tie
exactly. Problems count as tied when their scores are within `TIE_EPS` of each
other: sorted by $\theta$, each tie group starts at its highest-scoring problem
and takes every following problem whose $\theta$ is within `TIE_EPS` of that
first one, so no group spans more than `TIE_EPS`. Inside a group, $\theta$ no
longer matters: problems are ordered medium, easy, hard, then by a shuffle
seeded with `SEED`, so a rerun gives the same order. The `tie_group` column
shows which problems were tied.

</div>

## Attempting problems

Every attempting problems should appear in front of the list, i.e., steps should be:

1. Rank all attempting, put them on the list.
2. Rank all non-attempting, append to the list.

<div class="alert alert-block alert-info">

**Implementation note.** Not implemented, for the same reason as the *AC'ed* filter: "attempting" is the
per-user `status` field, which is `null` without that user's session cookie.

</div>

## Weights


```python
w1, w2, w3 = 20, 10, 1
```

> no reason, just guess.

## Ranking

In [8]:
DIFFICULTY_ORDER = {"Medium": 0, "Easy": 1, "Hard": 2}


def tie_groups(theta: pd.Series, eps: float) -> pd.Series:
    """
    Number the tie groups of `theta`, which must be sorted descending.

    A group starts at its first value and takes every following value within
    `eps` of it, so no group spans more than `eps`.
    """
    groups, group, head = [], 0, None
    for value in theta:
        if head is None or head - value >= eps:
            group += 1
            head = value
        groups.append(group)
    return pd.Series(groups, index=theta.index)


def candidates(problems: pd.DataFrame) -> pd.DataFrame:
    """The problems left after the command filters."""
    keep = pd.Series(True, index=problems.index)
    if not INCLUDE_PREMIUM:
        keep &= ~problems["premium"]
    if not INCLUDE_DATABASE:
        keep &= ~problems["tags"].map(lambda tags: "Database" in tags)
    return problems[keep]


def lift_scores(df: pd.DataFrame, f: dict[str, int]) -> pd.Series:
    """Mean lift of each problem's skill tags, scaled so the best problem is 1."""
    smoothed = {t: count + LIFT_SMOOTHING for t, count in f.items()}
    total_f = sum(smoothed.values())
    n = pd.Series([t for tags in df["skill_tags"] for t in tags]).value_counts()
    total_n = n.sum()
    lift = {t: (smoothed[t] / total_f) / (n[t] / total_n) for t in n.index}
    mean_lift = df["skill_tags"].map(
        lambda tags: float(np.mean([lift[t] for t in tags])) if tags else 0.0
    )
    return mean_lift / mean_lift.max()


def score(
    problems: pd.DataFrame,
    skills: pd.DataFrame,
    skill_score: str = "sum",
    solved_share: dict[str, float] | None = None,
) -> pd.DataFrame:
    """
    Score every candidate problem for one user, unsorted.

    `skill_score` is "sum" for the proposal's s_i or "lift" for the lift variant.
    `solved_share` maps each difficulty to the user's share of accepted problems;
    when given, c4 is added with weight W4.
    """
    f = dict(zip(skills["tag"], skills["f"]))
    total_f = sum(f.values())

    df = candidates(problems)[["title", "difficulty", "tags", "ac_rate", "length"]].copy()
    df["skill_tags"] = df["tags"].map(lambda tags: [t for t in tags if t in f])
    if skill_score == "lift":
        df["s"] = lift_scores(df, f)
    else:
        df["s"] = df["skill_tags"].map(lambda tags: sum(f[t] for t in tags)) / total_f
    df["c1"] = df["s"]
    df["c2"] = df["ac_rate"] / 100
    df["c3"] = 1 + M * df["length"].clip(upper=L)
    df["theta"] = W1 * df["c1"] + W2 * df["c2"] + W3 * df["c3"]
    if solved_share is not None:
        top_share = max(solved_share.values())
        df["c4"] = df["difficulty"].map(lambda d: solved_share[d] / top_share)
        df["theta"] += W4 * df["c4"]
    return df


def order(df: pd.DataFrame) -> pd.DataFrame:
    """Sort by theta, break ties within TIE_EPS, and number the ranks."""
    rng = np.random.default_rng(SEED)
    df = df.sort_values("theta", ascending=False)
    df["tie_group"] = tie_groups(df["theta"], TIE_EPS)
    df["_difficulty"] = df["difficulty"].map(DIFFICULTY_ORDER)
    df["_shuffle"] = rng.random(len(df))
    df = df.sort_values(["tie_group", "_difficulty", "_shuffle"])
    df = df.drop(columns=["_difficulty", "_shuffle"])
    df.insert(0, "rank", range(1, len(df) + 1))
    df.index.name = "id"
    return df


def rank(problems: pd.DataFrame, skills: pd.DataFrame) -> pd.DataFrame:
    """Score every candidate problem for one user and sort by theta."""
    return order(score(problems, skills))


rankings = {user: rank(problems, skills[user]) for user in skills}

COLUMNS = [
    "rank", "title", "difficulty", "theta", "tie_group", "s", "c1", "c2", "c3", "skill_tags"
]
print(f"{len(rankings[USERNAME])} candidate problems")
rankings[USERNAME][COLUMNS].head(TOP_N)

3300 candidate problems


,rank,title,difficulty,theta,tie_group,s,c1,c2,c3,skill_tags
id,,,,,,,,,,
1255,1,Maximum Score Words Formed by Letters,Hard,17.833,1,0.452,0.452,0.815,0.635,"[Array, Backtracking, Bit Manipulation, Dynamic Programm..."
1684,2,Count the Number of Consistent Strings,Easy,17.462,2,0.391,0.391,0.886,0.776,"[Array, Bit Manipulation, Hash Table, String]"
2418,3,Sort the People,Easy,17.068,3,0.390,0.390,0.848,0.791,"[Array, Hash Table, Sorting, String]"
1442,4,Count Triplets That Can Form Two Arrays of Equal XOR,Medium,16.842,4,0.375,0.375,0.848,0.859,"[Array, Bit Manipulation, Hash Table, Math]"
268,5,Missing Number,Easy,16.828,4,0.439,0.439,0.728,0.760,"[Array, Binary Search, Bit Manipulation, Hash Table, Mat..."
2125,6,Number of Laser Beams in a Bank,Medium,16.670,5,0.367,0.367,0.870,0.637,"[Array, Math, Matrix, String]"
1948,7,Delete Duplicate Folders in System,Hard,16.706,5,0.431,0.431,0.772,0.370,"[Array, Depth-First Search, Hash Table, Sorting, String,..."
1512,8,Number of Good Pairs,Easy,16.493,6,0.331,0.331,0.900,0.885,"[Array, Hash Table, Math]"
1048,9,Longest String Chain,Medium,16.435,7,0.470,0.470,0.636,0.672,"[Array, Dynamic Programming, Hash Table, Sorting, String..."


### Every problem's score

`rankings[USERNAME]` holds every candidate problem. The table above is its top
`TOP_N`; the cell below also writes it to `cache/ranking_<user>.csv` for
browsing in a spreadsheet.

In [9]:
for user, df in rankings.items():
    df.to_csv(CACHE_DIR / f"ranking_{user}.csv")

rankings[USERNAME][COLUMNS]

,rank,title,difficulty,theta,tie_group,s,c1,c2,c3,skill_tags
id,,,,,,,,,,
1255,1,Maximum Score Words Formed by Letters,Hard,17.833,1,0.452,0.452,0.815,0.635,"[Array, Backtracking, Bit Manipulation, Dynamic Programm..."
1684,2,Count the Number of Consistent Strings,Easy,17.462,2,0.391,0.391,0.886,0.776,"[Array, Bit Manipulation, Hash Table, String]"
2418,3,Sort the People,Easy,17.068,3,0.390,0.390,0.848,0.791,"[Array, Hash Table, Sorting, String]"
1442,4,Count Triplets That Can Form Two Arrays of Equal XOR,Medium,16.842,4,0.375,0.375,0.848,0.859,"[Array, Bit Manipulation, Hash Table, Math]"
268,5,Missing Number,Easy,16.828,4,0.439,0.439,0.728,0.760,"[Array, Binary Search, Bit Manipulation, Hash Table, Mat..."
...,...,...,...,...,...,...,...,...,...,...
4086,3296,Minimum Interval Selection to Cover Target Intervals,Hard,3.341,226,0.000,0.000,0.278,0.560,[]
3465,3297,Find Products with Valid Serial Numbers,Easy,3.272,227,0.002,0.002,0.284,0.401,[Database]
2286,3298,Booking Concert Tickets in Groups,Hard,3.187,228,0.036,0.036,0.205,0.417,"[Binary Indexed Tree, Binary Search, Design, Segment Tree]"


In [10]:
sizes = rankings[USERNAME].groupby("tie_group").size()
print(f"TIE_EPS={TIE_EPS}: {len(sizes)} tie groups, {(sizes > 1).sum()} with more than one problem")
top = rankings[USERNAME].head(TOP_N)
print(f"top {TOP_N}: {top['tie_group'].nunique()} tie groups")

TIE_EPS=0.05: 230 tie groups, 213 with more than one problem
top 20: 12 tie groups


### How much each criterion moves the score

The ranking depends on how far $w_j c_j$ *varies* across problems, not on its
size: a criterion that adds the same amount to every problem changes nothing.
The standard deviation of each weighted term is the useful number.

In [11]:
df = rankings[USERNAME]
weighted = pd.DataFrame(
    {"w1*c1": W1 * df["c1"], "w2*c2": W2 * df["c2"], "w3*c3": W3 * df["c3"]}
)
weighted.describe().loc[["mean", "std", "min", "max"]]

,w1*c1,w2*c2,w3*c3
mean,3.900,5.707,0.708
std,1.842,1.697,0.139
min,0.000,1.343,0.111
max,9.934,9.444,0.949


## Comparing users

### Top problems per user

In [12]:
pd.DataFrame(
    {user: df["title"].head(TOP_N).to_list() for user, df in rankings.items()},
    index=pd.RangeIndex(1, TOP_N + 1, name="rank"),
)

,xDDoubleTea,seanyu0425,wytsai7660
rank,,,
1,Maximum Score Words Formed by Letters,Maximum Score Words Formed by Letters,Sort the People
2,Count the Number of Consistent Strings,Sort the People,Maximum Score Words Formed by Letters
3,Sort the People,Count the Number of Consistent Strings,Delete Duplicate Folders in System
4,Count Triplets That Can Form Two Arrays of Equal XOR,Delete Duplicate Folders in System,Count the Number of Consistent Strings
5,Missing Number,Number of Laser Beams in a Bank,Longest String Chain
6,Number of Laser Beams in a Bank,Longest String Chain,Smallest String With Swaps
7,Delete Duplicate Folders in System,Missing Number,Missing Number
8,Number of Good Pairs,Find Maximum Number of String Pairs,Accounts Merge
9,Longest String Chain,Maximum Number of Coins You Can Get,Intersection of Two Arrays


### Top skill tags per user

In [13]:
pd.DataFrame(
    {
        user: (s["tag"] + " (" + s["f"].astype(str) + ")").head(10).to_list()
        for user, s in skills.items()
    },
    index=pd.RangeIndex(1, 11, name="#"),
)

,xDDoubleTea,seanyu0425,wytsai7660
#,,,
1,Array (101),Array (877),Array (208)
2,String (62),String (398),String (92)
3,Math (52),Hash Table (332),Hash Table (85)
4,Hash Table (48),Greedy (269),Sorting (76)
5,Tree (28),Math (263),Math (64)
6,Bit Manipulation (27),Sorting (223),Greedy (54)
7,Two Pointers (27),Dynamic Programming (172),Dynamic Programming (47)
8,Sorting (26),Two Pointers (126),Depth-First Search (45)
9,Binary Tree (26),Matrix (125),Two Pointers (42)


### Agreement between rankings

*Overlap* is the share of one user's top `TOP_N` that is also in the other's.
*Spearman* is the rank correlation of $\theta$ over all candidate problems: 1
means the same order, 0 means unrelated.

In [14]:
users = list(rankings)
theta = pd.DataFrame({user: rankings[user]["theta"] for user in users})
top = {user: set(rankings[user].index[:TOP_N]) for user in users}

rows = []
for i, a in enumerate(users):
    for b in users[i + 1 :]:
        rows.append(
            {
                "pair": f"{a} vs {b}",
                f"top {TOP_N} overlap": len(top[a] & top[b]) / TOP_N,
                "spearman (theta)": theta[a].rank().corr(theta[b].rank()),
            }
        )
pd.DataFrame(rows).set_index("pair")

,top 20 overlap,spearman (theta)
pair,,
xDDoubleTea vs seanyu0425,0.80,0.952
xDDoubleTea vs wytsai7660,0.70,0.966
seanyu0425 vs wytsai7660,0.85,0.994


## Modes: focus and mix

Both modes add the recent similarity $r_i$ (see [Recent AC](#Recent-AC)) to
$\theta_i$, with weight $w_r$:

$$
\theta_i^{\text{focus}} = \theta_i + w_r\, r_i
\qquad
\theta_i^{\text{mix}} = \theta_i + w_r\,(1 - r_i)
$$

- **Focus** suggests problems close to what the user just solved, to avoid the
  context switch that ends a session.
- **Mix** suggests problems unrelated to what the user just solved, for variety.
  Unrelated means unrelated to the last $K$ tagged problems, not the user's
  weakest tags.

The last 20 accepted problems are left out of both lists, since they are known
to be solved. Ties are broken as in the base ranking. `W_R` and `RECENT_K` are in
[Settings](#Settings).

In [15]:
RECENT_QUERY = """
query ($username: String!) {
  recentAcSubmissionList(username: $username, limit: 20) { titleSlug }
}
"""

TAGS_BY_SLUG = dict(zip(problems["slug"], problems["tags"]))
FRONTEND_ID_BY_SLUG = dict(zip(problems["slug"], problems.index))


def fetch_recent(username: str) -> list[str]:
    """Slugs of the user's last accepted problems, newest first."""
    submissions = gql(RECENT_QUERY, {"username": username})["recentAcSubmissionList"]
    slugs = []
    for submission in submissions:
        if submission["titleSlug"] not in slugs:
            slugs.append(submission["titleSlug"])
    return slugs


def recent_tagged(user: str) -> list[str]:
    """The user's last RECENT_K accepted problems that have tags."""
    return [s for s in recent[user] if TAGS_BY_SLUG.get(s)][:RECENT_K]


def jaccard(a: set, b: set) -> float:
    union = a | b
    return len(a & b) / len(union) if union else 0.0


def rank_mode(problems: pd.DataFrame, user: str, mode: str) -> pd.DataFrame:
    """Rank for one user in `focus` or `mix` mode."""
    recent_tags = [set(TAGS_BY_SLUG[s]) for s in recent_tagged(user)]
    solved = {FRONTEND_ID_BY_SLUG[s] for s in recent[user] if s in FRONTEND_ID_BY_SLUG}

    df = score(problems, skills[user])
    df = df[~df.index.isin(solved)].copy()
    df["r"] = df["tags"].map(
        lambda tags: float(np.mean([jaccard(set(tags), r) for r in recent_tags]))
        if recent_tags
        else 0.0
    )
    df["theta_base"] = df["theta"]
    bonus = df["r"] if mode == "focus" else 1 - df["r"]
    df["theta"] = df["theta_base"] + W_R * bonus
    return order(df)


recent = {user: fetch_recent(user) for user in skills}
modes = {
    (user, mode): rank_mode(problems, user, mode)
    for user in skills
    for mode in ("focus", "mix")
}

MODE_COLUMNS = ["rank", "title", "difficulty", "theta", "theta_base", "r", "tags"]

### The last `RECENT_K` tagged problems `USERNAME` solved

The table lists the last 20 accepted problems; `used` marks the ones $r_i$ is
computed from.

In [16]:
used = set(recent_tagged(USERNAME))
pd.DataFrame(
    [
        {
            "slug": slug,
            "id": FRONTEND_ID_BY_SLUG.get(slug),
            "tags": TAGS_BY_SLUG.get(slug),
            "used": slug in used,
        }
        for slug in recent[USERNAME]
    ]
)

,slug,id,tags,used
0,coupon-code-validator,3606,"[Array, Hash Table, Sorting, String]",True
1,minimum-path-sum,64,"[Array, Dynamic Programming, Matrix]",True
2,flipping-an-image,832,"[Array, Bit Manipulation, Matrix, Simulation, Two Pointers]",True
3,climbing-stairs-ii,3693,"[Array, Dynamic Programming]",True
4,check-if-all-1s-are-at-least-length-k-places-away,1437,[Array],True
5,subarrays-distinct-element-sum-of-squares-i,2913,"[Array, Binary Indexed Tree, Hash Table, Segment Tree]",False
6,maximum-alternating-sum-of-squares,3727,"[Array, Greedy, Sorting]",False
7,sum-of-squares-of-special-elements,2778,"[Array, Enumeration]",False
8,remove-outermost-parentheses,1021,"[Bracket Sequences, Stack, String]",False
9,linked-list-cycle,141,"[Floyd's Cycle Finding Algorithm, Hash Table, Linked Lis...",False


### Focus

In [17]:
modes[(USERNAME, "focus")][MODE_COLUMNS].head(TOP_N)

,rank,title,difficulty,theta,theta_base,r,tags
id,,,,,,,
2418,1,Sort the People,Easy,24.035,17.068,0.348,"[Array, Hash Table, Sorting, String]"
1277,2,Count Square Submatrices with All Ones,Medium,23.255,13.255,0.500,"[Array, Dynamic Programming, Matrix]"
1048,3,Longest String Chain,Medium,23.134,16.435,0.335,"[Array, Dynamic Programming, Hash Table, Sorting, String..."
118,4,Pascal's Triangle,Easy,23.044,12.911,0.507,"[Array, Dynamic Programming]"
49,5,Group Anagrams,Medium,22.935,15.968,0.348,"[Array, Hash Table, Sorting, String]"
1043,6,Partition Array for Maximum Sum,Medium,22.782,12.649,0.507,"[Array, Dynamic Programming]"
893,7,Groups of Special-Equivalent Strings,Medium,22.827,15.860,0.348,"[Array, Hash Table, Sorting, String]"
804,8,Unique Morse Code Words,Easy,22.729,16.024,0.335,"[Array, Hash Table, String]"
2125,9,Number of Laser Beams in a Bank,Medium,22.546,16.670,0.294,"[Array, Math, Matrix, String]"


### Mix

In [18]:
modes[(USERNAME, "mix")][MODE_COLUMNS].head(TOP_N)

,rank,title,difficulty,theta,theta_base,r,tags
id,,,,,,,
1261,1,Find Elements in a Contaminated Binary Tree,Medium,33.431,13.876,0.022,"[Binary Tree, Breadth-First Search, Depth-First Search, ..."
1255,2,Maximum Score Words Formed by Letters,Hard,33.383,17.833,0.222,"[Array, Backtracking, Bit Manipulation, Bitmask, Countin..."
1863,3,Sum of All Subset XOR Totals,Easy,33.178,16.249,0.154,"[Array, Backtracking, Bit Manipulation, Combinatorics, E..."
1028,4,Recover a Tree From Preorder Traversal,Hard,33.158,13.729,0.029,"[Binary Tree, Depth-First Search, String, Tree]"
1123,5,Lowest Common Ancestor of Deepest Leaves,Medium,32.926,13.290,0.018,"[Binary Lifting, Binary Tree, Breadth-First Search, DP o..."
654,6,Maximum Binary Tree,Medium,32.920,15.200,0.114,"[Array, Binary Tree, Cartesian Tree, Divide and Conquer,..."
1096,7,Brace Expansion II,Hard,32.825,14.539,0.086,"[Backtracking, Breadth-First Search, Hash Table, Sorting..."
865,8,Smallest Subtree with all the Deepest Nodes,Medium,32.743,13.106,0.018,"[Binary Lifting, Binary Tree, Breadth-First Search, DP o..."
1382,9,Balance a Binary Search Tree,Medium,32.676,12.676,0.000,"[Binary Search Tree, Binary Tree, Depth-First Search, Di..."


### Every user, both modes

*Shares a recent tag* is the share of the top `TOP_N` with at least one tag in
common with the user's last `RECENT_K` tagged problems: high in focus, low in mix.
*Mean r* is the mean recent similarity of the top `TOP_N`.

In [19]:
def recent_tag_set(user: str) -> set:
    return {t for s in recent_tagged(user) for t in TAGS_BY_SLUG[s]}


rows = []
for (user, mode), df in modes.items():
    top = df.head(TOP_N)
    tags = recent_tag_set(user)
    rows.append(
        {
            "user": user,
            "mode": mode,
            "shares a recent tag": top["tags"].map(lambda t: bool(tags & set(t))).mean(),
            "mean r": top["r"].mean(),
            "top 3": " | ".join(top["title"].head(3)),
        }
    )
pd.DataFrame(rows).set_index(["user", "mode"])

shares a recent tag  mean r                                                        top 3
user        mode                                                                                           
xDDoubleTea focus                 1.00   0.368  Sort the People | Count Square Submatrices with All Ones...
            mix                   0.75   0.071  Find Elements in a Contaminated Binary Tree | Maximum Sc...
seanyu0425  focus                 1.00   0.540  Pascal's Triangle | Partition Array for Maximum Sum | Ma...
            mix                   1.00   0.149  DI String Match | Sum of All Subset XOR Totals | Count t...
wytsai7660  focus                 1.00   0.302     Sort the People | Find Missing Elements | Group Anagrams
            mix                   1.00   0.133  Regions Cut By Slashes | Maximum Score Words Formed by L...

Mean $r$ separates the modes clearly (focus 0.30 – 0.54, mix 0.07 – 0.15 for
the three users), but *shares a recent tag* stays at 0.75 – 1.00 even in mix.
Array and String are on most problems, so almost every recent set contains one
of them, and the Jaccard similarity counts them like any other tag. Weighting
each tag by how rare it is across problems would let mix avoid the topic of the
session rather than these near-universal tags.

## Comparing variants: lift and $c_4$

Four variants of $\theta$, described in
[Personalization options](#Personalization-options):

| Variant | $c_1$ | $c_4$ |
|---|---|---|
| base | $s_i$ as proposed | no |
| $c_4$ | $s_i$ as proposed | yes, weight `W4` |
| lift | lift, smoothed by `LIFT_SMOOTHING` | no |
| lift + $c_4$ | lift, smoothed by `LIFT_SMOOTHING` | yes, weight `W4` |

$c_4$ needs each user's accepted count per difficulty, which is public
(`submitStats`).

In [20]:
SOLVED_QUERY = """
query ($username: String!) {
  matchedUser(username: $username) {
    submitStats { acSubmissionNum { difficulty count } }
  }
}
"""


def fetch_solved_share(username: str) -> dict[str, float]:
    """The user's share of accepted problems at each difficulty."""
    counts = gql(SOLVED_QUERY, {"username": username})["matchedUser"]["submitStats"]
    by_difficulty = {c["difficulty"]: c["count"] for c in counts["acSubmissionNum"]}
    return {d: by_difficulty[d] / by_difficulty["All"] for d in ("Easy", "Medium", "Hard")}


solved_share = {user: fetch_solved_share(user) for user in skills}
VARIANTS = {
    "base": {"skill_score": "sum", "c4": False},
    "c4": {"skill_score": "sum", "c4": True},
    "lift": {"skill_score": "lift", "c4": False},
    "lift + c4": {"skill_score": "lift", "c4": True},
}
variants = {
    (user, name): order(
        score(
            problems,
            skills[user],
            skill_score=spec["skill_score"],
            solved_share=solved_share[user] if spec["c4"] else None,
        )
    )
    for user in skills
    for name, spec in VARIANTS.items()
}
pd.DataFrame(solved_share).T.rename_axis("accepted share")

,Easy,Medium,Hard
accepted share,,,
xDDoubleTea,0.633,0.338,0.029
seanyu0425,0.443,0.472,0.084
wytsai7660,0.284,0.591,0.125


### Does each variant personalize?

Agreement between every pair of users under each variant. Lower overlap and
Spearman mean the users get more different lists.

In [21]:
users = list(skills)
rows = []
for name in VARIANTS:
    for a_i, a in enumerate(users):
        for b in users[a_i + 1 :]:
            da, db = variants[(a, name)], variants[(b, name)]
            theta = pd.DataFrame({"a": da["theta"], "b": db["theta"]}).dropna()
            rows.append(
                {
                    "variant": name,
                    "pair": f"{a} vs {b}",
                    f"top {TOP_N} overlap": len(set(da.index[:TOP_N]) & set(db.index[:TOP_N]))
                    / TOP_N,
                    "spearman": theta["a"].rank().corr(theta["b"].rank()),
                }
            )
agreement = pd.DataFrame(rows)
agreement.pivot(index="pair", columns="variant", values=[f"top {TOP_N} overlap", "spearman"])[
    [(m, v) for m in (f"top {TOP_N} overlap", "spearman") for v in VARIANTS]
]

top 20 overlap                       spearman                        
variant                             base    c4  lift lift + c4     base     c4   lift lift + c4
pair                                                                                           
seanyu0425 vs wytsai7660            0.85  0.65  0.25      0.25    0.994  0.871  0.862     0.775
xDDoubleTea vs seanyu0425           0.80  0.25  0.30      0.30    0.952  0.816  0.788     0.797
xDDoubleTea vs wytsai7660           0.70  0.00  0.70      0.35    0.966  0.492  0.838     0.507

### What each user gets

For each user and variant: the difficulty mix of the top `TOP_N`, how much of
it the base ranking also has in its top `TOP_N`, and the top 3. For `USERNAME`,
*already solved* is the share of the top `TOP_N` with an accepted submission in
the signed-in history.

In [22]:
accepted = (
    set(submissions.loc[submissions["status"] == "Accepted", "slug"]) if HAS_HISTORY else None
)
SLUG_BY_ID = dict(zip(problems.index, problems["slug"]))

rows = []
for (user, name), df in variants.items():
    top = df.head(TOP_N)
    base_top = set(variants[(user, "base")].index[:TOP_N])
    mix = top["difficulty"].value_counts()
    row = {
        "user": user,
        "variant": name,
        "E / M / H": f"{mix.get('Easy', 0)} / {mix.get('Medium', 0)} / {mix.get('Hard', 0)}",
        "overlap with base": len(set(top.index) & base_top) / TOP_N,
        "top 3": " | ".join(top["title"].head(3)),
    }
    if user == USERNAME and accepted is not None:
        row["already solved"] = top.index.map(lambda i: SLUG_BY_ID[i] in accepted).to_series().mean()
    rows.append(row)
pd.DataFrame(rows).set_index(["user", "variant"])

E / M / H  overlap with base                                                        top 3  already solved
user        variant                                                                                                              
xDDoubleTea base       10 / 7 / 3               1.00  Maximum Score Words Formed by Letters | Count the Number...            0.20
            c4         20 / 0 / 0               0.50  Count the Number of Consistent Strings | Sort the People...            0.30
            lift       7 / 12 / 1               0.00  Reverse Linked List | Swap Nodes in Pairs | Merge Two So...            0.30
            lift + c4  14 / 6 / 0               0.00  Reverse Linked List | Merge Two Sorted Lists | Remove Li...            0.35
seanyu0425  base        9 / 9 / 2               1.00  Maximum Score Words Formed by Letters | Sort the People ...             NaN
            c4         6 / 14 / 0               0.75  Sort the People | Number of Laser Beams in a Bank | Coun...             NaN
            lift       9 / 11 / 0               0.00  Root Equals Sum of Children | Search in a Binary Search ...             NaN
            lift + c4  5 / 15 / 0               0.00  Root Equals Sum of Children | Partitioning Into Minimum ...             NaN
wytsai7660  base       8 / 10 / 2               1.00  Sort the People | Maximum Score Words Formed by Letters ...             NaN
            c4         0 / 20 / 0               0.50  Longest String Chain | Smallest String With Swaps | Maxi...             NaN
            lift       5 / 14 / 1               0.00  Delete Node in a Linked List | Merge In Between Linked L...             NaN
            lift + c4  1 / 19 / 0               0.00  Delete Node in a Linked List | Merge In Between Linked L...             NaN

### `USERNAME`, lift + $c_4$

In [23]:
variants[(USERNAME, "lift + c4")][
    ["rank", "title", "difficulty", "theta", "s", "c2", "c3", "c4", "skill_tags"]
].head(TOP_N)

,rank,title,difficulty,theta,s,c2,c3,c4,skill_tags
id,,,,,,,,,
206,1,Reverse Linked List,Easy,39.007,1.000,0.812,0.891,1.000,"[Linked List, Recursion]"
21,2,Merge Two Sorted Lists,Easy,37.759,1.000,0.690,0.858,1.000,"[Linked List, Recursion]"
203,3,Remove Linked List Elements,Easy,36.462,1.000,0.557,0.891,1.000,"[Linked List, Recursion]"
83,4,Remove Duplicates from Sorted List,Easy,33.606,0.847,0.577,0.900,1.000,[Linked List]
24,5,Swap Nodes in Pairs,Medium,33.247,1.000,0.704,0.876,0.533,"[Linked List, Recursion]"
2236,6,Root Equals Sum of Children,Easy,32.400,0.653,0.852,0.824,1.000,"[Binary Tree, Tree]"
876,7,Middle of the Linked List,Easy,32.318,0.659,0.825,0.879,1.000,"[Linked List, Two Pointers]"
700,8,Search in a Binary Search Tree,Easy,32.259,0.653,0.832,0.880,1.000,"[Binary Tree, Tree]"
1669,9,Merge In Between Linked Lists,Medium,31.372,0.847,0.832,0.785,0.533,[Linked List]


### Reading the results

On 2026-10-18, with `W4 = 10` and `LIFT_SMOOTHING = 1`:

- **Lift + $c_4$ personalizes most.** Top-20 overlap between users falls from
  0.70 – 0.85 (base) to 0.25 – 0.35, and none of its top 20 is in the base top 20
  for any user.
- **$c_4$ alone collapses the difficulty.** `xDDoubleTea` gets 20 Easy and
  `wytsai7660` 20 Medium, because with `W4 = 10` the most-solved difficulty
  outweighs everything else. Combined with lift, the mix loosens (14 / 6 / 0
  and 1 / 19 / 0). A smaller `W4` would loosen it further. Hard leaves every
  top 20 that uses $c_4$.
- **Many suggestions are already solved,** and lift makes it worse: 20% of
  `xDDoubleTea`'s base top 20 has an accepted submission, 35% under lift + $c_4$.
  Lift favours the tags the user solves most, which is where their solved
  problems are, for example Reverse Linked List and Merge Two Sorted Lists. Without
  a signed-in session only the last 20 accepted problems are known, so for
  other users this cannot be measured or filtered.

## Personalization options

The rankings above barely differ between users, and a Hard problem can top the
list. This section records options for both, tested on `xDDoubleTea`,
`seanyu0425` and `wytsai7660` with the weights and filters above
($w_1, w_2, w_3 = 20, 10, 1$, premium excluded). Lift and $c_4$ are implemented in
[Comparing variants](#Comparing-variants:-lift-and-$c_4$); the Hard filter is
not.

### Why the rankings agree

Array, String and Hash Table are the top skill tags for all three users, because
they are the tags on the most problems. $s_i$ sums $f_k$ over a problem's tags,
so it mostly rewards problems carrying many common tags: every problem in the
top 10 has 4–6 tags, while a problem tagged only Array scores
$s_i = 101/608 = 0.17$ for `xDDoubleTea`.

Difficulty only breaks ties, so a Hard problem with a high $\theta$ keeps its
place.

### Results

The first exploration of these options, before smoothing was added to lift, is
superseded by [Comparing variants](#Comparing-variants:-lift-and-$c_4$), which
computes them from the current data on every run.

### Skill stats: lift

Compare each tag's share of the user's solves with its share of all problems:

$$
\mathrm{lift}_k = \dfrac{(f_k + \alpha) \,/\, \sum_{k'} (f_{k'} + \alpha)}{n_k \,/\, \sum_{k'} n_{k'}}
$$

where $n_k$ is the number of candidate problems tagged $t_k$, both sums run over
the tags in $T$, and $\alpha$ is `LIFT_SMOOTHING`. The tag score is the mean lift over the problem's skill
tags, scaled so the best problem scores 1:

$$
s_i = \dfrac{\operatorname{mean}_{t_k \in T_i} \mathrm{lift}_k}{\max_{i'} \operatorname{mean}_{t_k \in T_{i'}} \mathrm{lift}_k}
$$

with $T_i$ the skill tags of problem $i$, and $s_i = 0$ when $T_i$ is empty.

- Tags a user solves more often than LeetCode's mix score above 1, so the list
  follows the user: for `xDDoubleTea` the top problems become Linked List ones,
  and the top-20 overlap with `seanyu0425` drops from 0.80 to 0.25.
- The mean removes the bonus for carrying many tags.
- A rare tag solved once would get a large lift from one data point
  (`xDDoubleTea` has Rolling Hash ×1). Adding $\alpha$ to every count shrinks
  lifts built on few solves toward 1.
- Lift favours what the user already does, which is the intent: see
  [Staying close to what the user knows](#Staying-close-to-what-the-user-knows) above.

### Difficulty: $c_4$ from the solved mix

$$
c_4(i) = \dfrac{a_{d_i}}{\max_d a_d}
$$

where $a_d$ is the user's share of accepted problems at difficulty $d$
(`submitStats.acSubmissionNum`, public) and $d_i$ is the difficulty of problem
$i$. Tested with $w_4 = 10$.

- Hard leaves every user's top 20 with no filter, since Hard is the smallest
  share for all three.
- The list follows the user's level: `xDDoubleTea` (63% Easy) gets 15 Easy in
  the top 20, `wytsai7660` (59% Medium) gets 19 Medium.
- It keeps users at their current level. This is acceptable: see
  [Staying close to what the user knows](#Staying-close-to-what-the-user-knows) above.

### Difficulty: Hard as a filter

Add Hard to the command filters, next to Premium and Database, and rank Easy and
Medium by default. It is the simplest option and matches the existing filters.
Excluded by default, it hides Hard from users who solve many (`seanyu0425` has
129), so it works best as a switch on top of $c_4$.

## Do switches end sessions?

[Staying close to what the user knows](#Staying-close-to-what-the-user-knows)
rests on one claim: switching to an unrelated topic makes a session more likely
to end. This section checks it against `USERNAME`'s own submission history.

The history needs a signed-in LeetCode session: put the cookies in `.env` and
run `uv run leetcode_auth.py`, which writes `cache/submissions.json` (see that
file's docstring). Without it, the cells below skip.

This notebook is committed with its outputs to a public repository, so
anything printed from the history becomes public. `leetcode_auth.py` never
fetches submitted code.

### Definitions

- A **session** is a run of submissions with no gap longer than $g$ minutes;
  $g$ takes each value in `SESSION_GAPS_MIN`.
- A **visit** is one problem within one session, from its first submission
  there.
- A visit **follows** the previous visit in the same session. Only visits where
  both problems are tagged are compared.
- A visit is a **switch** when it shares no tag with the previous one. The
  *ignoring Array/String* variant drops those two tags first, since they are on
  most problems.
- A visit **failed** when it has a non-accepted submission before its first
  accepted one, or no accepted one at all.
- A visit **ends the session** when it is the session's last.

Each comparison is a difference of two proportions, switch minus related. The
p-value is a two-sided permutation test: the switch labels are shuffled
`PERMUTATIONS` times, and $p$ is the share of shuffles whose difference is at
least as large as the observed one.

In [24]:
COMMON_TAGS = {"Array", "String"}


def visits(submissions: pd.DataFrame, gap_min: int) -> pd.DataFrame:
    """One row per problem per session, in order."""
    df = submissions.copy()
    df["session"] = (df["timestamp"].diff() > gap_min * 60).cumsum()
    failed = lambda status: bool(
        (status.reset_index(drop=True) != "Accepted").cumprod().sum()
    )
    v = (
        df.groupby(["session", "slug"], sort=False)
        .agg(start=("timestamp", "min"), failed=("status", failed))
        .reset_index()
        .sort_values(["session", "start"], ignore_index=True)
    )
    v["tags"] = v["slug"].map(lambda s: frozenset(TAGS_BY_SLUG.get(s) or []))
    v["difficulty"] = v["slug"].map(DIFFICULTY_BY_SLUG)
    v["position"] = v.groupby("session").cumcount()
    v["ends_session"] = v["position"] == v.groupby("session")["position"].transform("max")
    v["previous_tags"] = v.groupby("session")["tags"].shift()
    return v


def transitions(v: pd.DataFrame, ignore_common: bool) -> pd.DataFrame:
    """Visits that follow a tagged problem in the same session, labelled switch or not."""
    t = v[(v["position"] > 0) & v["previous_tags"].notna()].copy()
    drop = COMMON_TAGS if ignore_common else set()
    t["switch"] = [
        not ((a - drop) & (b - drop)) for a, b in zip(t["tags"], t["previous_tags"])
    ]
    keep = [bool(a) and bool(b) for a, b in zip(t["tags"], t["previous_tags"])]
    return t[keep]


def permutation_p(values: pd.Series, labels: pd.Series, rng) -> tuple[float, float]:
    """Observed difference in means (labelled minus not) and its two-sided p-value."""
    values = values.to_numpy(dtype=float)
    labels = labels.to_numpy(dtype=bool)
    observed = values[labels].mean() - values[~labels].mean()
    shuffled = np.empty(PERMUTATIONS)
    for k in range(PERMUTATIONS):
        perm = rng.permutation(labels)
        shuffled[k] = values[perm].mean() - values[~perm].mean()
    p = (np.sum(np.abs(shuffled) >= abs(observed)) + 1) / (PERMUTATIONS + 1)
    return observed, p


if HAS_HISTORY:
    rng = np.random.default_rng(SEED)
    rows = []
    for gap in SESSION_GAPS_MIN:
        v = visits(submissions, gap)
        sizes = v.groupby("session").size()
        for ignore_common in (False, True):
            t = transitions(v, ignore_common)
            row = {
                "gap (min)": gap,
                "switch means": "no shared tag, ignoring Array/String"
                if ignore_common
                else "no shared tag",
                "sessions": len(sizes),
                "sessions with 2+ problems": int((sizes >= 2).sum()),
                "n related": int((~t["switch"]).sum()),
                "n switch": int(t["switch"].sum()),
            }
            for outcome in ("failed", "ends_session"):
                diff, p = permutation_p(t[outcome], t["switch"], rng)
                row[f"{outcome}: related"] = t.loc[~t["switch"], outcome].mean()
                row[f"{outcome}: switch"] = t.loc[t["switch"], outcome].mean()
                row[f"{outcome}: p"] = p
            rows.append(row)
    session_results = pd.DataFrame(rows).set_index(["gap (min)", "switch means"])
    display(session_results)

sessions  sessions with 2+ problems  n related  n switch  failed: related  failed: switch  failed: p  ends_session: related  ends_session: switch  \
gap (min) switch means                                                                                                                                                                              
15        no shared tag                              239                         49         50        34            0.480           0.294      0.116                  0.640                 0.500   
          no shared tag, ignoring Array/String       239                         49         36        48            0.500           0.333      0.180                  0.639                 0.542   
30        no shared tag                              192                         57         61        48            0.492           0.354      0.181                  0.574                 0.458   
          no shared tag, ignoring Array/String       192                         57         44        65            0.523           0.369      0.118                  0.568                 0.492   
60        no shared tag                              164                         55         70        61            0.457           0.377      0.379                  0.443                 0.393   
          no shared tag, ignoring Array/String       164                         55         50        81            0.480           0.383      0.282                  0.520                 0.358   

                                                ends_session: p  
gap (min) switch means                                           
15        no shared tag                                   0.267  
          no shared tag, ignoring Array/String            0.509  
30        no shared tag                                   0.256  
          no shared tag, ignoring Array/String            0.560  
60        no shared tag                                   0.598  
          no shared tag, ignoring Array/String            0.073

### Difficulty

Harder problems could cause both failures and session ends, so the comparison
is also split by difficulty, at a 30-minute gap.

In [25]:
if HAS_HISTORY:
    t = transitions(visits(submissions, 30), ignore_common=False)
    t["kind"] = np.where(t["switch"], "switch", "related")
    display(
        t.groupby(["difficulty", "kind"]).agg(
            n=("slug", "size"),
            failed=("failed", "mean"),
            ends_session=("ends_session", "mean"),
        )
    )

n  failed  ends_session
difficulty kind                             
Easy       related  43   0.419         0.558
           switch   39   0.385         0.436
Hard       switch    1   0.000         0.000
Medium     related  18   0.667         0.611
           switch    8   0.250         0.625

### Reading the results

On 2026-10-18, with 777 submissions since 2023-08, none of the comparisons
supported the claim: after a switch, `xDDoubleTea` failed less often and ended
the session less often, and no difference reached $p < 0.05$. Three limits keep
this from refuting it either:

- **The outcome that matters is invisible.** Opening a problem after a switch,
  losing track and leaving without submitting leaves no submission, so those
  visits are missing from this data. The bot sees them: `/problem` and the
  thread select log every problem opened. Recording `(user, problem, time)` per
  open would make "opened, never submitted" measurable.
- **The sample is small.** Most sessions have one problem, so only about a
  hundred transitions exist; a 10-point difference in proportions needs several
  hundred to detect.
- **Tag overlap is a rough measure of relatedness,** and most transitions are
  between Easy problems, so difficulty cannot be controlled for.